In [ ]:

# Generates the guess set W_k and solution set S_k for an alternative version of Wordle
# with words of length k, and plots the (normalized) frequency distribution of W_k with
# the words of S_k marked, mirroring Section 3.3-3.4 of the report.

import numpy as np
import math
from matplotlib import pyplot as plt
import itertools

length = 11  # set the desired word length k

# Load the English dictionary and its word frequencies (aligned line by line)
eng_dic = np.genfromtxt('English_dict_words.txt', delimiter='\n', dtype=str)
word_freq = np.genfromtxt('English_dict_frequencies.txt', delimiter='\n', dtype=str)  # stored as strings
dic_freq = np.vstack((eng_dic, word_freq))

# Keep only the words of the chosen length, then sort by frequency (descending)
ind = [ind for ind, i in enumerate(eng_dic) if len(i) != length]
dic_len7 = np.delete(dic_freq, ind, axis=1)
print(dic_len7.shape[1])
freq_numbers = [eval(i) for i in dic_len7[1]]
i = np.argsort(freq_numbers)[::-1]
dic_len7 = dic_len7[:, i]


# ---------------------------------------------------------------------
# Construct the set of valid guesses W_k (Section 3.3.1)
# ---------------------------------------------------------------------

dist = np.array([0, 11, 205, 821, 1734, 2752, 3403, 0, 0, 4046])  # distribution of L in the original game
array_ind = np.zeros(10, int)   # number of length-k words in each frequency interval
print("The distribution of all the {}-letter words is as follows: \n".format(length))
freq = [eval(i) for i in dic_len7[1]]
List_ind = []  # indices of the words belonging to each interval
for k in range(1, 11):
    ind = [x for x, i in enumerate(freq) if i <= 10 ** -(k) and i > 10 ** (-(k + 1))]
    array_ind[k - 1] = len(ind)
    List_ind.append(ind)
print(array_ind)

# Use the algorithm from 3.3.1 only when there are more length-k words than in L (5 <= k <= 14)
if length >= 5 and length <= 14:
    ind_new_dict = [[] for i in range(10)]
    for i in range(len(dist)):
        t = i
        while dist[i] != 0:
            if dist[i] > array_ind[t]:
                ran_ind = List_ind[t].copy()
                ind_new_dict[t] = [j for k in [ind_new_dict[t], ran_ind] for j in k]
                dist[i] = dist[i] - len(ran_ind)
                List_ind[t] = []
                array_ind[t] = 0
                t = t + 1
                continue
            if dist[i] <= array_ind[t]:
                ran_ind = np.random.choice(List_ind[t], dist[i], replace=False)
                dist[i] = 0
                array_ind[t] = array_ind[t] - len(ran_ind)
                List_ind[t] = [elt for elt in List_ind[t] if elt not in ran_ind]
                ind_new_dict[t] = [j for k in [ind_new_dict[t], ran_ind] for j in k]

    # distribution of the newly-constructed dictionary
    dist_guesses = [0 for i in range(10)]
    for i in range(len(ind_new_dict)):
        dist_guesses[i] = len(ind_new_dict[i])
    print("The distribution of all the {}-letter words in our dictionary is as follows: ".format(length))
    print(dist_guesses)

    # flatten the chosen indices and keep only those words
    ind_newdict_array = np.array(list(itertools.chain(
        ind_new_dict[0], ind_new_dict[1], ind_new_dict[2], ind_new_dict[3], ind_new_dict[4],
        ind_new_dict[5], ind_new_dict[6], ind_new_dict[7], ind_new_dict[8], ind_new_dict[9])))
    indices_not_dictionary = [i for i in range(dic_len7.shape[1]) if i not in ind_newdict_array]
    dic_len7 = np.delete(dic_len7, indices_not_dictionary, axis=1)
    freq_numbers = [eval(i) for i in dic_len7[1]]
    i = np.argsort(freq_numbers)[::-1]
    dic_len7 = dic_len7[:, i]

# if length < 5 or length > 14, W_k is simply all length-k words in the dictionary (already in dic_len7)


# ---------------------------------------------------------------------
# Construct the set of solutions S_k (Sections 3.3.2 - 3.3.4)
# ---------------------------------------------------------------------

dist = np.array([0, 11, 168, 532, 963, 595, 40, 0, 0, 0])  # distribution of S in the original game
array_ind = np.zeros(10, int)
print("The distribution of the solutions is as follows: ")
freq = [eval(i) for i in dic_len7[1]]
List_ind = []
for k in range(1, 11):
    ind = [x for x, i in enumerate(freq) if i <= 10 ** -(k) and i > 10 ** (-(k + 1))]
    array_ind[k - 1] = len(ind)
    List_ind.append(ind)

freq_setguesses = [eval(i) for i in dic_len7[1]]
Sum = np.sum(freq_setguesses)
freq_setguesses = freq_setguesses * 1 / Sum
Area_required = 0.80822   # normalized area contributed by S in the original game
Area_curve = 0
threshold = 0.02

ind_new_dict = [[] for i in range(10)]

# Case: length < 5 or length > 14  (algorithm 3.3.4)
if length < 5 or length > 14:
    percentages_list = [0, 1.0, 0.8195121951219512, 0.6479902557856273, 0.5553633217993079,
                         0.2162063953488372, 0.011754334410813987, 0, 0, 0.0]
    t = 1
    for i in range(len(dist)):
        if List_ind[i] == []:
            ind_new_dict[i] = []
            continue
        if i == 0:
            ran_ind = List_ind[i].copy()
            ind_new_dict[i] = [j for k in [ind_new_dict[i], ran_ind] for j in k]
            if length == 4:
                continue
            t += 1
        else:
            if i > 0 and i < 7:
                number_elt_add = math.floor(len(List_ind[i]) * percentages_list[t])
                ran_ind = np.random.choice(List_ind[i], number_elt_add, replace=False)
                ind_new_dict[i] = [j for k in [ind_new_dict[i], ran_ind] for j in k]
                t += 1

    ind_newdict_array = np.array(list(itertools.chain(
        ind_new_dict[0], ind_new_dict[1], ind_new_dict[2], ind_new_dict[3], ind_new_dict[4],
        ind_new_dict[5], ind_new_dict[6], ind_new_dict[7], ind_new_dict[8], ind_new_dict[9])))
    indices_not_dictionary = [i for i in range(dic_len7.shape[1]) if i not in ind_newdict_array]
    dic_len7_sol = np.delete(dic_len7, indices_not_dictionary, axis=1)
    freq_numbers = [eval(i) for i in dic_len7_sol[1]]
    i = np.argsort(freq_numbers)[::-1]
    dic_len7_sol = dic_len7_sol[:, i]

    # trim words from the end until the normalized area is within `threshold` of Area_required
    word_frequencies_sol = [eval(i) for i in dic_len7_sol[1]]
    norml_frequencies_sol = word_frequencies_sol.copy()
    norml_frequencies_sol = norml_frequencies_sol * 1 / Sum
    Area_sol = np.sum(norml_frequencies_sol)

    if Area_sol - Area_required > threshold:
        while Area_sol - Area_required > threshold:
            word_frequencies_sol = word_frequencies_sol[:-2]
            norml_frequencies_sol = norml_frequencies_sol[:-2]
            dic_len7_sol = np.delete(dic_len7_sol, [dic_len7_sol.shape[1] - 1, dic_len7_sol.shape[1] - 2], 1)
            Area_sol = np.sum(norml_frequencies_sol)

# Case: 5 <= length <= 14  (algorithm 3.3.3)
if length >= 5 and length <= 14:
    for i in range(len(dist)):
        t = i
        while dist[i] != 0:
            if dist[i] > array_ind[t]:
                ran_ind = List_ind[t].copy()
                ind_new_dict[t] = [j for k in [ind_new_dict[t], ran_ind] for j in k]
                dist[i] = dist[i] - len(ran_ind)
                List_ind[t] = []
                array_ind[t] = 0
                t = t + 1
                continue
            if dist[i] <= array_ind[t]:
                ran_ind = np.random.choice(List_ind[t], dist[i], replace=False)
                dist[i] = 0
                array_ind[t] = array_ind[t] - len(ran_ind)
                List_ind[t] = [elt for elt in List_ind[t] if elt not in ran_ind]
                ind_new_dict[t] = [j for k in [ind_new_dict[t], ran_ind] for j in k]

    ind_newdict_array = np.array(list(itertools.chain(
        ind_new_dict[0], ind_new_dict[1], ind_new_dict[2], ind_new_dict[3], ind_new_dict[4],
        ind_new_dict[5], ind_new_dict[6], ind_new_dict[7], ind_new_dict[8], ind_new_dict[9])))
    indices_not_dictionary = [l for l in range(dic_len7.shape[1]) if l not in ind_newdict_array]
    dic_len7_sol = np.delete(dic_len7, indices_not_dictionary, axis=1)
    freq_numbers = [eval(l) for l in dic_len7_sol[1]]
    ind = np.argsort(freq_numbers)[::-1]
    dic_len7_sol = dic_len7_sol[:, ind]

    word_frequencies_sol = [eval(ind) for ind in dic_len7_sol[1]]
    norml_frequencies_sol = word_frequencies_sol.copy()
    norml_frequencies_sol = norml_frequencies_sol * 1 / Sum
    Area_sol = np.sum(norml_frequencies_sol)
    while Area_sol - Area_required > threshold:
        word_frequencies_sol = word_frequencies_sol[:-2]
        norml_frequencies_sol = norml_frequencies_sol[:-2]
        dic_len7_sol = np.delete(dic_len7_sol, [dic_len7_sol.shape[1] - 1, dic_len7_sol.shape[1] - 2], 1)
        Area_sol = np.sum(norml_frequencies_sol)

array_ind = np.zeros(10, int)
for k in range(1, 11):
    ind = [i for i in word_frequencies_sol if i <= 10 ** -(k) and i > 10 ** (-(k + 1))]
    array_ind[k - 1] = len(ind)
print(array_ind)

print("The size of the set of solutions is {}".format(dic_len7_sol.shape[1]))


# ---------------------------------------------------------------------
# Plots (Section 3.4)
# ---------------------------------------------------------------------

# W_k with the words of S_k marked
xpoints = np.arange(start=0, stop=len(dic_len7[0]), step=1)
markers_on = [ind for ind, i in enumerate(dic_len7[0]) if i in dic_len7_sol[0]]
plt.plot(xpoints, freq_setguesses)
plt.plot(xpoints, freq_setguesses, '-bx', markevery=markers_on, label='line with select markers')
plt.yscale('log')
plt.title('Distribution frequencies of the set of all possible guesses (Length = {})'.format(length))
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.show()
print("The area under the curve is 1 and the area of the elements in the set of solutions is {:.5}."
      .format(Area_sol))

# Top-18%-most-common W_k words vs S_k
percentage = math.floor(dic_len7.shape[1] * 0.18)
dic_2309 = np.delete(dic_len7, slice(percentage, dic_len7.shape[1]), 1)
intersection = np.intersect1d(dic_2309[0], dic_len7_sol[0])
size_inter_2309 = np.shape(intersection)[0]
print("{} words from the set of solutions are in the top 18% most common words "
      "(there are {} words in this top).".format(size_inter_2309, math.floor(dic_2309.shape[1])))

xpoints = np.arange(start=0, stop=len(dic_2309[1]), step=1)
freq_setsol = [eval(i) for i in dic_2309[1]]
Sum = np.sum(freq_setsol)
freq_setsol = freq_setsol * 1 / Sum
freq_inter = [eval(i) for i in dic_len7[1][0:size_inter_2309 - 1]]
freq_inter = freq_inter * 1 / Sum
Area = np.sum(freq_inter)
plt.plot(xpoints, freq_setsol, 'b', np.arange(start=0, stop=size_inter_2309 - 1, step=1), freq_inter, 'r')
plt.yscale('log')
plt.fill_between(np.arange(start=0, stop=size_inter_2309 - 1, step=1), freq_inter, alpha=0.7)
plt.title('Distribution frequencies of the set of solutions (Length = {})'.format(length))
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.show()
print("The area under the curve is 1 and the area of the blue region is {:.4}".format(Area))


# ---------------------------------------------------------------------
# Sort alphabetically and export
# ---------------------------------------------------------------------

i = np.argsort(dic_len7[0])[::1]
dic_len7 = dic_len7[:, i]
i = np.argsort(dic_len7_sol[0])[::1]
dic_len7_sol = dic_len7_sol[:, i]
print(dic_len7)

# Generate txt files (uncomment to write to disk)
# np.savetxt("dict_letterword.txt", dic_len7[0], fmt='%10.20s', delimiter=" ")          # W_k, alphabetical
# np.savetxt("freq_letterword.txt", dic_len7[1], fmt='%10.20s', delimiter=" ")          # frequencies of W_k
# np.savetxt("dict_letterword_sol.txt", dic_len7_sol[0], fmt='%10.20s', delimiter=" ")  # S_k, alphabetical
# np.savetxt("freq_letterword_sol.txt", dic_len7_sol[1], fmt='%10.20s', delimiter=" ")  # frequencies of S_k
